In [0]:
%run ./data_splitter_nb

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col


from pyspark.sql.functions import col


class CategoricalEncoder:

    def __init__(self, categorical_columns):
        self.categorical_columns = categorical_columns
        self.ohe_columns = [c + "_ohe" for c in categorical_columns]
        self.fitted_pipeline = None
        self.category_labels = {}   # stores actual category names per column, in index order

    def build_pipeline(self):
        indexers = [
            StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep")
            for c in self.categorical_columns
        ]
        encoders = [
            OneHotEncoder(inputCol=c + "_index", outputCol=c + "_ohe")
            for c in self.categorical_columns
        ]
        return Pipeline(stages=indexers + encoders)

    def fit(self, train_df):
        pipeline = self.build_pipeline()
        self.fitted_pipeline = pipeline.fit(train_df)

        # Extract category labels from each fitted StringIndexerModel, in index order
        for c in self.categorical_columns:
            indexer_model = self.fitted_pipeline.stages[self.categorical_columns.index(c)]
            self.category_labels[c] = indexer_model.labels   # e.g. ["Gasoline", "Ethylene", "Bitumen", ...]

        print("Encoder fitted on training data only")

    def transform(self, df):
        if self.fitted_pipeline is None:
            raise ValueError("Call fit() before transform().")

        transformed_df = self.fitted_pipeline.transform(df)

        # Convert each OHE vector into separate named columns
        for c in self.categorical_columns:
            ohe_col = c + "_ohe"
            array_col = vector_to_array(col(ohe_col))
            labels = self.category_labels[c]

            for i, label in enumerate(labels):
                safe_label = label.replace(" ", "_")
                transformed_df = transformed_df.withColumn(
                    c + "_" + safe_label,
                    array_col[i]
                )

            transformed_df = transformed_df.drop(ohe_col)   # drop the combined array column now that it's expanded

        return transformed_df


class PreprocessedDataWriter:

    def __init__(self, spark, schema_name):
        self.spark = spark
        self.schema_name = schema_name

    def write(self, df, table_name):
        full_table_name = self.schema_name + "." + table_name
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table_name)
        print("Written: " + full_table_name)


# ---------------------------
# Notebook usage
# ---------------------------

categorical_columns = ["product_name", "destination_city"]   # confirm this list matches your actual categorical columns

# Step 1: Fit encoder ONLY on train_df
encoder = CategoricalEncoder(categorical_columns=categorical_columns)
encoder.fit(train_df)

# Step 2: Transform all three sets using the SAME fitted encoder (no re-fitting, no leakage)
train_encoded_df = encoder.transform(train_df)
validation_encoded_df = encoder.transform(validation_df)
test_encoded_df = encoder.transform(test_df)

# Step 3: Save to ML schema
writer = PreprocessedDataWriter(spark=spark, schema_name="oil_gas_mlops.ml_data")

writer.write(train_encoded_df, "train_data")
writer.write(validation_encoded_df, "validation_data")
writer.write(test_encoded_df, "test_data")